# CyberGPT V1.1 — Generación controlada de texto con modelos GPT
**Mini-proyecto NLP / Universidad ICESI** · Caso original: redactar borradores técnicos de alertas de vulnerabilidades a partir del catálogo público CISA KEV.

**Pregunta de investigación:** ¿cómo afectan *temperature*, *top-p*, *top-k* y las instrucciones estructuradas a la fidelidad factual, la diversidad y la latencia de los textos generados por un modelo autoregresivo?

**Alcance:** generación condicionada, **no** RAG/MCP; las fichas se entregan explícitamente al modelo en el prompt. Un borrador generado no reemplaza una evaluación de seguridad. No se fabrican valores CVSS ni parches no documentados.

**Evidencias esperadas:** EDA, tabla de experimentos, 5+ gráficas, ejemplos contrastados, rúbrica humana, exportaciones y demo. Los resultados se generan al ejecutar el notebook; no se presentan cifras experimentales inventadas.

**Alineación con notebook de la sesión:** logits/softmax, generación incremental epsilon-greedy, comparación con `model.generate` y ajuste fino opcional de GPT-2 español. El corpus de chistes del profesor NO se reutiliza.

## 0. Entorno limpio y dependencias
Abrir un **nuevo** Colab con GPU T4 (si está disponible). Instalar solo lo necesario; **no** Ollama, LangChain, MCP ni Gradio por obligación. Evitar mezclar con los notebooks anteriores. Si se modifica `transformers`, Colab puede solicitar reiniciar Python una vez. El análisis del dataset funciona también con CPU.

In [ ]:
%pip -q install "transformers>=4.48,<5" "accelerate>=1,<2" "pandas>=2.1,<3" "numpy>=1.26,<3" "matplotlib>=3.7" "scikit-learn>=1.3,<2" "requests>=2.31" "datasets>=3,<5"

In [ ]:
import os, time, re, json, random, importlib.metadata as mdlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import torch
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('cybergpt_resultados'); OUT.mkdir(exist_ok=True)
print('Torch:',torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
for p in ['transformers','accelerate','pandas']:
    print(p,mdlib.version(p))

## 1. Definición del problema y datos públicos
Los analistas necesitan convertir hallazgos públicos, redactados de manera heterogénea, en textos breves, uniformes y rastreables. El principal riesgo de la generación libre es **inventar hechos**. Elegimos KEV porque contiene CVE identificables, descripciones y acciones de mitigación. La fuente es [CISA KEV](https://www.cisa.gov/known-exploited-vulnerabilities-catalog), con espejo oficial [cisagov/kev-data](https://github.com/cisagov/kev-data).

El campo `knownRansomwareCampaignUse` describe si hay asociación conocida; no permite afirmar ausencia de explotación. `dueDate` es un vencimiento aplicable al marco de referencia del catálogo, no una fecha universal para cualquier organización.

In [ ]:
URLS=[
 'https://raw.githubusercontent.com/cisagov/kev-data/develop/known_exploited_vulnerabilities.json',
 'https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json']
raw=None; used_url=None
for u in URLS:
    try:
        r=requests.get(u,timeout=35); r.raise_for_status(); tmp=r.json()
        if not isinstance(tmp.get('vulnerabilities'),list): raise ValueError('Formato KEV no reconocido')
        raw=tmp; used_url=u; break
    except Exception as e: print('Fuente no disponible:',u, '|', str(e)[:100])
if raw is None: raise RuntimeError('No se descargó KEV. Verifica Internet y reejecuta esta celda.')
df=pd.DataFrame(raw['vulnerabilities'])
assert {'cveID','vendorProject','product','vulnerabilityName','shortDescription','requiredAction','dateAdded'}.issubset(df.columns)
assert df['cveID'].is_unique
print('Fuente:',used_url,'| Versión:',raw.get('catalogVersion'),'| Filas:',len(df))
df.to_csv(OUT/'catalogo_kev_snapshot.csv',index=False)
(OUT/'metadata_dataset.json').write_text(json.dumps({'source':used_url,'catalogVersion':raw.get('catalogVersion'),'dateReleased':raw.get('dateReleased'),'rows':len(df)},indent=2),encoding='utf-8')
display(df.head(3))

## 2. Análisis exploratorio (EDA)
Mediremos cobertura, duplicados, longitud de descripciones, proveedores, evolución anual y etiquetas de ransomware. Los campos vacíos deben describirse, no completarse con supuestos.

In [ ]:
print('Dimensiones:',df.shape,'| CVE duplicados:',df.cveID.duplicated().sum())
missing=df.isna().sum().sort_values(ascending=False); display(missing.to_frame('faltantes').head(12))
df['dateAdded_dt']=pd.to_datetime(df['dateAdded'],errors='coerce')
df['year']=df['dateAdded_dt'].dt.year
df['description_chars']=df['shortDescription'].fillna('').astype(str).str.len()
df['action_chars']=df['requiredAction'].fillna('').astype(str).str.len()
fig,axs=plt.subplots(2,2,figsize=(14,9))
df['vendorProject'].value_counts().head(12).sort_values().plot.barh(ax=axs[0,0],title='Top 12 proveedores')
df['year'].value_counts().sort_index().plot.bar(ax=axs[0,1],title='Entradas KEV por año de incorporación')
df['description_chars'].hist(ax=axs[1,0],bins=35); axs[1,0].set_title('Longitud de descripciones (caracteres)')
df.get('knownRansomwareCampaignUse',pd.Series(dtype=str)).fillna('No informado').value_counts().plot.bar(ax=axs[1,1],title='Etiqueta de ransomware (literal KEV)')
for ax in axs.flat: ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT/'eda_4_paneles.png',dpi=160); plt.show()
print('Mediana caracteres descripción:',df['description_chars'].median())

## 3. Muestreo y ficha factual
Usaremos fichas completas y reproducibles. La generación se basa **solo** en los campos mostrados. La evaluación automática posterior verifica CVE, proveedor y citas literales; no debe confundirse con verificación semántica completa.

In [ ]:
POOL=df.dropna(subset=['cveID','shortDescription','requiredAction']).copy()
POOL=POOL[(POOL.shortDescription.astype(str).str.len()>40)&(POOL.requiredAction.astype(str).str.len()>20)]
sample=POOL.sample(n=min(12,len(POOL)),random_state=SEED).reset_index(drop=True)
assert len(sample)>=3
def ficha(r):
    campos=['cveID','vendorProject','product','vulnerabilityName','shortDescription','requiredAction','dateAdded','dueDate','knownRansomwareCampaignUse','notes']
    return '\n'.join(f'{k}: {r.get(k,"No informado")}' for k in campos)
print(ficha(sample.iloc[0]))
sample.to_csv(OUT/'muestra_experimental.csv',index=False)

## 4. Modelo autoregresivo de la familia GPT
Se elige **Qwen2.5-0.5B-Instruct**: es un modelo **decoder-only** generativo (estilo GPT) ligero, con soporte en español e instrucciones. **No es GPT-2 ni un producto de OpenAI**. Para una comparación opcional y más literal con GPT-2, se incluye un experimento separado al final.

Se mide en GPU T4 si está disponible; usar CPU es posible pero lento. Modelos pequeños también pueden alucinar. El primer tiempo incluye descarga/carga.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
t0=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,torch_dtype=(torch.float16 if torch.cuda.is_available() else torch.float32),device_map='auto' if torch.cuda.is_available() else None)
model.eval()
print('Modelo cargado:',MODEL_ID,'| tiempo:',round(time.perf_counter()-t0,1),'s | dispositivo:',next(model.parameters()).device)

In [ ]:
SYSTEM=('Eres un redactor técnico prudente de ciberseguridad. Usa EXCLUSIVAMENTE la ficha suministrada. '
 'No inventes puntuaciones CVSS, versiones afectadas, fechas, explotación, parches ni instrucciones técnicas no presentes. '
 'Escribe en español. Si un detalle no está en la ficha, escribe "No especificado en la ficha". '
 'Evita convertir un campo de fecha en una obligación universal para todas las empresas.')
def construir_prompt(row, formato='estructurado'):
    dato=ficha(row)
    if formato=='libre':
        instruccion='Redacta una explicación breve de la vulnerabilidad, máximo 100 palabras.'
    else:
        instruccion=('Redacta una alerta breve, máximo 120 palabras, con las secciones: '
                     'CVE, Producto/proveedor, Descripción, Acción recomendada, Fuente. '
                     'Cita el identificador CVE. Si algo falta, dilo claramente.')
    msgs=[{'role':'system','content':SYSTEM}, {'role':'user','content':f'FICHA PÚBLICA CISA KEV:\n{dato}\n\nTAREA: {instruccion}'}]
    return tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
def generar(row, temperatura=0.2, top_p=0.9, top_k=50, max_nuevos=130, formato='estructurado', estrategia='muestreo'):
    prompt=construir_prompt(row,formato)
    inputs=tokenizer(prompt,return_tensors='pt',truncation=True,max_length=1600).to(next(model.parameters()).device)
    kwargs={'max_new_tokens':int(max_nuevos),'pad_token_id':tokenizer.eos_token_id,'do_sample':estrategia=='muestreo'}
    if estrategia=='muestreo':kwargs.update(temperature=float(temperatura),top_p=float(top_p),top_k=int(top_k))
    if torch.cuda.is_available(): torch.cuda.synchronize()
    tic=time.perf_counter()
    with torch.inference_mode(): out=model.generate(**inputs,**kwargs)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    elapsed=time.perf_counter()-tic
    tokens=out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(tokens,skip_special_tokens=True).strip(),elapsed,len(tokens),inputs['input_ids'].shape[1]

## 5. Prueba funcional de generación
Antes de las comparaciones, verificar que el modelo cargado produzca un texto. La respuesta NO se considerará automáticamente correcta: contrastar con la ficha de origen.

In [ ]:
fila=sample.iloc[0]
resp,seg,nt,nprompt=generar(fila,max_nuevos=120,estrategia='greedy')
print('CVE:',fila['cveID'],'| Tiempo:',round(seg,2),'s | Tokens entrada/salida:',nprompt,nt)
print(resp)

## 5A. Inspección de logits y probabilidades de siguiente token (alineación con guía ICESI)

El cuaderno de la sesión muestra cómo `model(...).logits` proporciona puntuaciones para el siguiente token y cómo `softmax` las convierte en probabilidades. Aquí usamos una **frase sobre una alerta CVE**, no «Había una vez». La probabilidad siguiente no representa confianza factual sobre una vulnerabilidad; solo preferencias del modelo ante el texto suministrado.

In [ ]:
from IPython.display import display
entrada = 'Alerta de ciberseguridad: se ha identificado la vulnerabilidad'
tok = tokenizer(entrada, return_tensors='pt').to(model.device)
with torch.inference_mode():
    logits = model(**tok).logits[0, -1, :]
    probs = torch.softmax(logits.float(), dim=-1)
valores, indices = probs.topk(10)
top_tokens = pd.DataFrame({'token': [tokenizer.decode([int(i)]) for i in indices], 'probabilidad': valores.cpu().numpy()})
display(top_tokens)
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(top_tokens.token[::-1].astype(str),top_tokens.probabilidad[::-1]);ax.set_title('Top-10 siguientes tokens tras un prefijo de alerta');ax.set_xlabel('Probabilidad de siguiente token');plt.tight_layout();plt.show()
print('Forma de logits:', tuple(logits.shape), 'Suma de probabilidades:', float(probs.sum()))

## 5B. Generación incremental propia: epsilon-greedy

Reimplementamos la lógica de la función `generate` del profesor, con dos mejoras: acumulación correcta de `input_ids` y corte con token EOS. Con `eps=1` se escoge siempre el máximo; con `eps=0` se muestrea; en valores intermedios se alterna probabilísticamente. Para la comparación se usa un prefijo del dominio de ciberseguridad y solo 35 tokens de salida para evitar esperas largas.

In [ ]:
def generar_epsilon(prefijo, eps=1.0, pasos=35, seed=42):
    assert 0 <= eps <= 1
    rng = np.random.default_rng(seed)
    ids = tokenizer(prefijo, return_tensors='pt')['input_ids'].to(model.device)
    filas = []
    with torch.inference_mode():
        for paso in range(pasos):
            logits = model(input_ids=ids).logits[0,-1].float()
            p = torch.softmax(logits, dim=-1)
            top_p, top_i = torch.topk(p,3)
            siguiente = torch.argmax(p).reshape(1,1) if rng.random() < eps else torch.multinomial(p, 1).reshape(1,1)
            filas.append({'paso':paso+1,'tipo':'greedy' if int(siguiente)==int(torch.argmax(p)) else 'muestreo',
                         'token_elegido':tokenizer.decode([int(siguiente)]),
                         'p_elegido':float(p[int(siguiente)].item()),
                         'candidatos':' | '.join(f'{tokenizer.decode([int(j)])}:{float(v):.3f}' for v,j in zip(top_p,top_i))})
            ids = torch.cat([ids,siguiente.to(ids.device)],dim=1)
            if int(siguiente) == tokenizer.eos_token_id: break
    return tokenizer.decode(ids[0],skip_special_tokens=True),pd.DataFrame(filas)

prefijo='Boletín de seguridad: la vulnerabilidad CVE requiere revisar'
trazas=[]
for eps in [1.0,0.5,0.0]:
    texto,traza=generar_epsilon(prefijo,eps=eps,pasos=35)
    trazas.append({'eps':eps,'texto':texto,'pasos':len(traza),'prob_media_token':traza.p_elegido.mean()})
    print(f'\nEPS={eps}: {texto[:600]}')
    if eps==0.5: display(traza.head(8))
df_epsilon=pd.DataFrame(trazas)
display(df_epsilon[['eps','pasos','prob_media_token']])

## 5C. Fine-tuning opcional de un GPT-2 español con boletines públicos

El ejemplo del docente adapta `mrm8488/spanish-gpt2` a chistes. Aquí, para tener un experimento distinto y comparable, usamos **descripciones y acciones de CISA KEV** como corpus. Se entrena un GPT-2 separado del Qwen usado para la demo, evitando alterar el generador principal. Es **opcional**: consume tiempo de GPU y puede generar memorización; nunca debe entrenarse con información privada. La comparación antes/después será cualitativa y con pérdida de validación (no se inferirá veracidad a partir de menor loss).

In [ ]:
# Configuración del experimento opcional: cambiar a True solamente si deseas entrenar.
EJECUTAR_FINE_TUNING = False
N_TRAIN = 160
EPOCAS = 1
if EJECUTAR_FINE_TUNING:
    from datasets import Dataset
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
    gpt_id='mrm8488/spanish-gpt2'
    tok_gpt=AutoTokenizer.from_pretrained(gpt_id)
    if tok_gpt.pad_token is None: tok_gpt.pad_token=tok_gpt.eos_token
    modelo_gpt=AutoModelForCausalLM.from_pretrained(gpt_id)
    modelo_gpt.config.pad_token_id=tok_gpt.pad_token_id
    corpus=(df.dropna(subset=['shortDescription','requiredAction'])
              .assign(text=lambda x: 'Boletín: '+x.cveID.astype(str)+'. '+x.shortDescription.astype(str)+'. Acción: '+x.requiredAction.astype(str)))
    corpus=corpus.sample(n=min(N_TRAIN,len(corpus)),random_state=42)[['text']]
    datos=Dataset.from_pandas(corpus,preserve_index=False).train_test_split(test_size=0.15,seed=42)
    def tokenizar(batch): return tok_gpt(batch['text'],truncation=True,max_length=128)
    datos=datos.map(tokenizar,batched=True,remove_columns=['text'])
    args=TrainingArguments(output_dir='cybergpt_gpt2_finetuned',num_train_epochs=EPOCAS,
        per_device_train_batch_size=2,per_device_eval_batch_size=2,gradient_accumulation_steps=4,
        learning_rate=2e-5,eval_strategy='epoch',save_strategy='no',report_to='none',
        logging_steps=10,fp16=torch.cuda.is_available())
    trainer=Trainer(model=modelo_gpt,args=args,train_dataset=datos['train'],eval_dataset=datos['test'],
        data_collator=DataCollatorForLanguageModeling(tokenizer=tok_gpt,mlm=False))
    metricas_antes=trainer.evaluate()
    pref='Boletín de ciberseguridad:'
    def muestra_gpt2():
        ids=tok_gpt(pref,return_tensors='pt').input_ids.to(modelo_gpt.device)
        with torch.inference_mode(): salida=modelo_gpt.generate(ids,max_new_tokens=65,do_sample=False,pad_token_id=tok_gpt.eos_token_id)
        return tok_gpt.decode(salida[0],skip_special_tokens=True)
    muestra_antes=muestra_gpt2()
    trainer.train()
    metricas_despues=trainer.evaluate()
    muestra_despues=muestra_gpt2()
    print('Loss validación antes:',metricas_antes['eval_loss'],'después:',metricas_despues['eval_loss'])
    print('ANTES:',muestra_antes,'\nDESPUÉS:',muestra_despues)
else:
    print('Fine-tuning OMITIDO por defecto. Cambia EJECUTAR_FINE_TUNING=True para ejecutar la comparación opcional.')

### Interpretación de resultados conforme al notebook guía

- `eps=1`: determinismo en generación greedy, si el entorno y modelo permanecen iguales.
- `eps=0`: mayor variabilidad posible; realizar varias semillas antes de afirmar diferencias.
- El gráfico de los próximos tokens describe el espacio probabilístico del modelo, **no** la gravedad real de las CVE.
- Fine-tuning: comparar pérdida de validación y dos muestras antes/después; no concluir mejora general con pocos ejemplos.
- El caso original utiliza CISA KEV en lugar del dataset de chistes del notebook guía.

## 6. Diseño experimental: parámetros y estrategias
Se comparan **greedy** y muestreo con tres temperaturas y dos top-p, sobre las mismas tres fichas. La semilla controla aleatoriedad dentro de lo posible; variaciones de GPU pueden impedir reproducción bit a bit. Limitar ejecución en Colab: 3 fichas × 5 configuraciones = 15 generaciones.

**Hipótesis:** greedy y temperaturas bajas podrían reducir variación, pero no garantizan fidelidad. Una diversidad mayor tampoco implica mejor calidad.

In [ ]:
CONFIGS=[
 {'nombre':'greedy','estrategia':'greedy','temperatura':0.2,'top_p':1.0,'top_k':50},
 {'nombre':'temp_02','estrategia':'muestreo','temperatura':0.2,'top_p':0.9,'top_k':50},
 {'nombre':'temp_07','estrategia':'muestreo','temperatura':0.7,'top_p':0.9,'top_k':50},
 {'nombre':'temp_10','estrategia':'muestreo','temperatura':1.0,'top_p':0.9,'top_k':50},
 {'nombre':'top_p_06','estrategia':'muestreo','temperatura':0.7,'top_p':0.6,'top_k':50}
]
print('Generaciones planeadas:',len(CONFIGS)*3)

In [ ]:
# Ejecutar después de la prueba funcional; en Colab T4 puede tardar varios minutos.
records=[]
for j in range(3):
    row=sample.iloc[j]
    for config in CONFIGS:
        torch.manual_seed(SEED+j)
        respuesta,elapsed,nout,nin=generar(row,max_nuevos=110,**{k:v for k,v in config.items() if k!='nombre'})
        records.append({'cveID':row.cveID,'config':config['nombre'],'strategy':config['estrategia'],
                        'temp':config['temperatura'],'top_p':config['top_p'],
                        'latencia_s':round(elapsed,3),'tokens_entrada':nin,'tokens_salida':nout,
                        'tokens_por_seg':round(nout/max(elapsed,1e-9),2),'respuesta':respuesta,
                        'cve_presente':row.cveID.lower() in respuesta.lower(),
                        'proveedor_presente':str(row.vendorProject).lower() in respuesta.lower(),
                        'palabras':len(respuesta.split())})
results=pd.DataFrame(records)
results.to_csv(OUT/'resultados_generacion.csv',index=False)
display(results[['cveID','config','latencia_s','tokens_por_seg','cve_presente','proveedor_presente','palabras']])

## 7. Métricas y gráficas comparativas
Métricas automáticas **proxy**: presencia textual de CVE y proveedor, longitud, latencia y tokens/s. Ni CVE presente ni proveedor presente demuestran que el contenido sea correcto. Complementar con auditoría humana de hechos, sobreafirmaciones y pertinencia.

In [ ]:
assert len(results)>0,'+'"Ejecuta antes la celda experimental"
summary=results.groupby('config',as_index=False).agg(latencia_media=('latencia_s','mean'),tps_media=('tokens_por_seg','mean'),cve_rate=('cve_presente','mean'),proveedor_rate=('proveedor_presente','mean'),palabras_media=('palabras','mean'))
display(summary.round(2)); summary.to_csv(OUT/'resumen_metricas.csv',index=False)
fig,axs=plt.subplots(2,2,figsize=(13,8))
axs[0,0].bar(summary.config,summary.latencia_media);axs[0,0].set_title('Latencia media por configuración (s)')
axs[0,1].bar(summary.config,summary.tps_media);axs[0,1].set_title('Velocidad media (tokens/s)')
axs[1,0].bar(summary.config,summary.cve_rate);axs[1,0].set_ylim(0,1.05);axs[1,0].set_title('Fracción con CVE explícito (proxy)')
axs[1,1].bar(summary.config,summary.palabras_media);axs[1,1].set_title('Extensión media (palabras)')
for ax in axs.flat:ax.tick_params(axis='x',rotation=25);ax.grid(alpha=.2)
plt.tight_layout();plt.savefig(OUT/'comparacion_decodificacion.png',dpi=160);plt.show()

In [ ]:
# Variabilidad de longitud y latencia por ficha y configuración
pivot=results.pivot(index='cveID',columns='config',values='latencia_s')
fig,ax=plt.subplots(figsize=(9,4))
im=ax.imshow(pivot.values,aspect='auto');ax.set_xticks(range(len(pivot.columns)),pivot.columns,rotation=20);ax.set_yticks(range(len(pivot.index)),pivot.index)
ax.set_title('Mapa de calor — latencia por CVE y configuración');fig.colorbar(im,ax=ax,label='segundos')
plt.tight_layout();plt.savefig(OUT/'mapa_calor_latencias.png',dpi=160);plt.show()

## 8. Evaluación cualitativa y riesgo de alucinación
Se proporcionan pares **ficha vs. texto generado**. **No** asignar puntuaciones automáticas a exactitud semántica a partir de coincidencias simples. La rúbrica puntúa 0–2 por criterio (0=incumple, 1=parcial, 2=cumple): hechos soportados, acción correcta, claridad, no introducir datos y utilidad. Una respuesta que inventa un parche/versiones/CVSS debe recibir penalización.

In [ ]:
comparar=results[['cveID','config','respuesta']].copy()
for idx,row in comparar.head(5).iterrows():
    print('\n'+'='*70,'\n',row.cveID,'|',row.config,'\n',row.respuesta[:1200])
rubrica=pd.DataFrame([{'cveID':x.cveID,'config':x.config,'hechos_0_2':None,'accion_0_2':None,'claridad_0_2':None,'sin_invenciones_0_2':None,'utilidad_0_2':None,'observaciones':''} for x in results.itertuples()])
rubrica.to_csv(OUT/'plantilla_rubrica_humana.csv',index=False)
print('Plantilla creada:',OUT/'plantilla_rubrica_humana.csv')

## 9. Experimento adicional: misma ficha, prompt libre vs. estructurado
Aísla el efecto del **prompt** con decoding greedy fijo. Una ficha; dos prompts. Examinar si secciones y advertencias reducen omisiones, sin atribuir causalidad con esta muestra pequeña.

In [ ]:
prompt_tests=[]
for modo in ['libre','estructurado']:
    texto,lat,tok,_=generar(sample.iloc[1],formato=modo,estrategia='greedy',max_nuevos=140)
    prompt_tests.append({'formato':modo,'latencia_s':lat,'tokens':tok,'texto':texto})
    print('\n',modo,'—',round(lat,2),'s\n',texto)
pd.DataFrame(prompt_tests).to_csv(OUT/'experimento_prompt.csv',index=False)

## 10. Demo interactiva opcional en Gradio
Para evitar conflictos de paquetes, Gradio no se instala en la celda principal. Ejecutar esta sección solo si necesitas una demo interactiva o grabar un video; el notebook principal ya cumple generación de texto, EDA y experimentación. La app **no es un RAG ni agente MCP**: el usuario selecciona el CVE de la muestra y el modelo redacta a partir de la ficha. La interfaz genera un enlace temporal con `share=True`.

In [ ]:
# Opcional: ejecutar si deseas interfaz Gradio
%pip -q install "gradio>=5.35,<6" "huggingface-hub>=0.34,<1"

In [ ]:
import gradio as gr
subset={str(r.cveID):r for _,r in sample.iterrows()}
def demo_alerta(cve,temperatura):
    if cve not in subset:return 'CVE no encontrado en la muestra.'
    texto,seg,nt,_=generar(subset[cve],temperatura=float(temperatura),max_nuevos=120)
    return f'{texto}\n\n---\nFuente: CISA KEV · {cve} · {used_url}\nLatencia: {seg:.2f} s · Tokens: {nt}'
with gr.Blocks(title='CyberGPT — Generador de alertas') as demo:
    gr.Markdown('# CyberGPT — Alertas técnicas basadas en CISA KEV\nGeneración condicionada; verificar las salidas contra la ficha original.')
    with gr.Row():cve=gr.Dropdown(choices=list(subset),value=list(subset)[0],label='CVE de la muestra');temp=gr.Slider(.1,1.2,value=.2,step=.1,label='Temperatura')
    boton=gr.Button('Generar alerta'); salida=gr.Markdown()
    boton.click(demo_alerta,[cve,temp],salida)
demo.queue(default_concurrency_limit=1).launch(share=True,debug=False)

## 11. Interpretación y conclusiones — completar tras ejecutar
Al terminar, describir: (1) versión del dataset; (2) proporción de campos faltantes; (3) configuración con menor mediana de latencia; (4) fidelidad evaluada **manualmente**; (5) ejemplos de omisión o invención; (6) diferencias entre decoding y prompting; (7) límites del tamaño muestral.

**Amenazas a la validez:** 3 CVE no representan todo KEV; las respuestas son generadas bajo distintas cargas GPU; proxies de coincidencia literal no capturan veracidad; la curación cambia cuando KEV se actualiza. La muestra se exporta para reproducibilidad.

**Etica y seguridad:** usar únicamente fuentes públicas; no difundir datos de activos internos, direcciones, credenciales ni instrucciones de explotación. Revisar humanamente cualquier alerta antes de uso operativo.

In [ ]:
# Exportación final para GitHub/entrega
import zipfile
archive='CyberGPT_V1_resultados_colab.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir():
        if f.is_file():z.write(f,arcname=f.name)
print('Archivo de resultados:',archive,'| archivos:',[x.name for x in OUT.iterdir()])